In [ ]:
# train_qlora_sft.py
"""
Fine-tunes Llama 3.1 8B using QLoRA and SFTTrainer for hardware bug detection.

Assumes input dataset is a .jsonl file where each line is a JSON object
with a 'completion' field containing another JSON object with 'buggy_code',
'description', 'justification', and 'metadata' containing 'cwe_id'.

The script reformats this data to train a bug detector model.
"""

import os
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    logging,
)
from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training
from trl import SFTTrainer
import argparse
import json

# --- Configuration --- (Adjust these paths and parameters as needed)

# Model details
BASE_MODEL_NAME = "meta-llama/Llama-3.1-8B" # Model to fine-tune

# Dataset details
# Default path, can be overridden by CLI argument
DEFAULT_DATASET_PATH = '/home/aseth7/hwsec/bug_insertion/data/synthetic_sft_dataset_structured.jsonl'

# Output directory for trained model adapters
DEFAULT_OUTPUT_DIR = "./results/llama3_1_8b_detector_qlora"

# QLoRA parameters
LORA_R = 64                # LoRA attention dimension (rank)
LORA_ALPHA = 16            # Alpha parameter for LoRA scaling
LORA_DROPOUT = 0.1         # Dropout probability for LoRA layers
USE_4BIT = True            # Activate 4-bit precision base model loading
BNB_4BIT_COMPUTE_DTYPE = "float16" # Compute dtype for 4-bit base models
BNB_4BIT_QUANT_TYPE = "nf4"       # Quantization type (fp4 or nf4)
USE_NESTED_QUANT = False   # Activate nested quantization for 4-bit base models (double quantization)

# Training arguments
NUM_TRAIN_EPOCHS = 1              # Number of training epochs (start with 1-3 for QLoRA)
PER_DEVICE_TRAIN_BATCH_SIZE = 4   # Batch size per GPU for training
PER_DEVICE_EVAL_BATCH_SIZE = 4    # Batch size per GPU for evaluation
GRADIENT_ACCUMULATION_STEPS = 1   # Number of update steps to accumulate gradients over
LEARNING_RATE = 2e-4              # Initial learning rate (AdamW optimizer)
WEIGHT_DECAY = 0.001              # Weight decay to apply (if not zero)
OPTIMIZER = "paged_adamw_32bit"   # Optimizer to use
LR_SCHEDULER_TYPE = "cosine"      # Learning rate schedule type
MAX_GRAD_NORM = 0.3               # Maximum gradient norm (for gradient clipping)
WARMUP_RATIO = 0.03               # Ratio of steps for linear warmup phase
GROUP_BY_LENGTH = True            # Group sequences into batches with similar lengths - speeds up training
SAVE_STEPS = 50                   # Save checkpoint every X updates steps
LOGGING_STEPS = 25                # Log every X updates steps
MAX_SEQ_LENGTH = 2048             # Maximum sequence length to use (adjust based on data and memory)


# --- Dataset Formatting Function ---

def format_detector_prompt(sample):
    """
    Formats a sample from the synthetic dataset for detector training.

    Input: A dictionary from the loaded dataset.
    Output: A dictionary {'text': formatted_string}
            where formatted_string follows the Llama 3 instruction template.
    """
    try:
        completion_data = sample.get("completion", {})
        metadata = sample.get("metadata", {})

        buggy_code = completion_data.get("buggy_code")
        description = completion_data.get("description")
        justification = completion_data.get("justification")
        cwe_id = metadata.get("cwe_id")

        # Basic validation
        if not all([buggy_code, description, justification, cwe_id]):
            # print(f"Warning: Skipping sample due to missing data: {sample}")
            return {"text": None} # Return None for text to filter later

        # --- Construct the Prompt for the Model ---
        # This is what the *user* would ask the detector model
        input_prompt = f"Analyze the following Verilog RTL code for hardware security vulnerabilities. Identify the CWE ID, describe the vulnerability, explain the justification, and pinpoint the vulnerable line(s) marked with a comment like '# {cwe_id}'.\n\nBEGIN RTL CODE:\n{buggy_code}\nEND RTL CODE"

        # --- Construct the Target Completion for the Model ---
        # This is the *ideal response* we want the model to learn
        # We don't have precise line numbers for synthetic code easily,
        # so we instruct the model to find the comment marker.
        # The model's output during inference should ideally follow this structure.
        target_completion = f"""**Analysis:**
1.  **Vulnerability:** {description}
2.  **Location:** Vulnerability related to the line(s) marked with '# {cwe_id}'.
3.  **Explanation:** {justification}
4.  **CWE ID:** {cwe_id}"""

        # --- Combine using Llama 3 Instruct Template ---
        formatted_text = f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n{input_prompt}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n{target_completion}<|eot_id|>"

        return {"text": formatted_text}

    except Exception as e:
        print(f"Error formatting sample: {e}\nSample: {sample}")
        return {"text": None}


# --- Main Training Function ---

def train(dataset_path, output_dir):
    """Loads data, configures model/trainer, and runs SFT."""

    print(f"Loading dataset from: {dataset_path}")
    # Load the dataset; assumes it's a .jsonl file
    try:
        dataset = load_dataset("json", data_files=dataset_path, split="train")
        print(f"Dataset loaded: {dataset}")
    except Exception as e:
        print(f"Error loading dataset: {e}")
        return

    print("Formatting dataset for detector training...")
    original_columns = dataset.column_names
    dataset = dataset.map(format_detector_prompt, remove_columns=original_columns)

    # Filter out samples that failed formatting
    dataset = dataset.filter(lambda example: example["text"] is not None)
    print(f"Dataset size after formatting and filtering: {len(dataset)}")
    if len(dataset) == 0:
        print("Error: No valid samples remaining after formatting. Check dataset structure and formatting function.")
        return

    # --- Load Tokenizer and Model with QLoRA Configuration ---
    print(f"Loading base model: {BASE_MODEL_NAME}")

    compute_dtype = getattr(torch, BNB_4BIT_COMPUTE_DTYPE)

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=USE_4BIT,
        bnb_4bit_quant_type=BNB_4BIT_QUANT_TYPE,
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_use_double_quant=USE_NESTED_QUANT,
    )

    # Check GPU compatibility with bfloat16
    if compute_dtype == torch.float16 and USE_4BIT:
        major, _ = torch.cuda.get_device_capability()
        if major >= 8:
            print("=" * 80)
            print("Your GPU supports bfloat16: accelerate training with bf16=True")
            print("=" * 80)

    # Load base model
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_NAME,
        quantization_config=bnb_config,
        # device_map="auto" # Automatically distributes model across GPUs
        device_map={"": 0} # Or explicitly place on GPU 0
    )
    model.config.use_cache = False # Required for gradient checkpointing
    model.config.pretraining_tp = 1 # Set if pretraining tensor parallelism > 1

    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, trust_remote_code=True)
    # Llama 3 specific padding setup
    tokenizer.pad_token = tokenizer.eos_token # Use EOS token for padding
    tokenizer.padding_side = "right" # Pad on the right

    print("Preparing model for K-bit training...")
    model = prepare_model_for_kbit_training(model)

    # --- PEFT Configuration ---
    peft_config = LoraConfig(
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        r=LORA_R,
        bias="none",
        task_type="CAUSAL_LM",
        # Specify target modules for Llama 3 (check model architecture if needed)
        target_modules=[
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
            "gate_proj",
            "up_proj",
            "down_proj",
            # "lm_head", # Sometimes included
        ],
    )

    # --- Training Arguments ---
    print("Setting up training arguments...")
    training_arguments = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=NUM_TRAIN_EPOCHS,
        per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        optim=OPTIMIZER,
        save_steps=SAVE_STEPS,
        logging_steps=LOGGING_STEPS,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        fp16=False, # fp16/bf16 disabled for 4-bit training unless specifically enabled
        bf16=False, # Set to True if using an Ampere or newer GPU and BNB_4BIT_COMPUTE_DTYPE is "bfloat16"
        max_grad_norm=MAX_GRAD_NORM,
        # max_steps=-1, # Limit number of training steps if desired
        warmup_ratio=WARMUP_RATIO,
        group_by_length=GROUP_BY_LENGTH,
        lr_scheduler_type=LR_SCHEDULER_TYPE,
        report_to="tensorboard", # Or "wandb" if configured
        gradient_checkpointing=True, # Enable gradient checkpointing to save memory
        # ddp_find_unused_parameters=False, # Set if using DDP
    )

    # --- Initialize SFTTrainer ---
    print("Initializing SFTTrainer...")
    trainer = SFTTrainer(
        model=model,
        train_dataset=dataset,
        peft_config=peft_config,
        dataset_text_field="text", # The column containing our formatted prompts/completions
        max_seq_length=MAX_SEQ_LENGTH,
        tokenizer=tokenizer,
        args=training_arguments,
        packing=False, # Set to True if you want to pack multiple sequences into one sample
    )

    # --- Start Training ---
    print("Starting training...")
    train_result = trainer.train()

    # --- Save Training Metrics and Final Model ---
    metrics = train_result.metrics
    trainer.log_metrics("train", metrics)
    trainer.save_metrics("train", metrics)

    print(f"Saving final adapter model to {output_dir}")
    trainer.save_model(output_dir) # Saves the LoRA adapter weights

    # Clean up GPU memory
    del model
    del trainer
    torch.cuda.empty_cache()
    gc.collect()

    print("Training finished successfully!")


# --- Script Execution ---
if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Fine-tune Llama 3.1 8B with QLoRA for bug detection.")
    parser.add_argument(
        "--dataset_path",
        type=str,
        default=DEFAULT_DATASET_PATH,
        help="Path to the synthetic dataset JSONL file."
    )
    parser.add_argument(
        "--output_dir",
        type=str,
        default=DEFAULT_OUTPUT_DIR,
        help="Directory to save the trained LoRA adapters."
    )
    # Add more CLI arguments to override config parameters if desired (e.g., epochs, batch_size)

    args = parser.parse_args()

    # Ensure output directory exists
    os.makedirs(args.output_dir, exist_ok=True)

    train(args.dataset_path, args.output_dir)